In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Memeriksa ulang 200 label pilot dengan pedoman v1**

In [6]:
import os
import pandas as pd
from IPython.display import display

os.makedirs("data/processed", exist_ok=True)

processed_path = "data/processed/comments_v0_audited.csv"
posts_b2_path = "data/raw/posts_b2.csv"
labels_batch2_path = "data/processed/labels_student_v1(b3).csv"

if os.path.exists(processed_path):
    df_comments = pd.read_csv(processed_path)
    df_eligible = df_comments[df_comments["alasan_eksklusi"] == "Lolos"].copy()

    if "text_content" in df_eligible.columns:
        df_eligible["text_original"] = df_eligible["text_content"]

    # Buat kerangka DataFrame baru HANYA dengan kolom yang diwajibkan panduan
    kolom_wajib = [
        "comment_id", "post_id", "context_text", "text_original",
        "context_target_phrase", "sentiment_label", "reference_label", "annotation_note"
    ]

    # Isi dengan string kosong untuk kolom yang belum ada
    for col in kolom_wajib:
        if col not in df_eligible.columns:
            df_eligible[col] = ""

    # Filter hanya kolom wajib
    df_bersih = df_eligible[kolom_wajib].copy()
    df_bersih = df_bersih.sort_values(by="post_id").reset_index(drop=True)

    # HANYA ambil Batch 3 (baris 400 sampai 600)
    df_batch3 = df_bersih.iloc[400:600].copy()

    # Tarik context_text khusus untuk Batch 3
    if os.path.exists(posts_b2_path):
        df_posts = pd.read_csv(posts_b2_path)
        post_text_col = next((c for c in ['text', 'caption', 'content', 'post_text', 'text_content'] if c in df_posts.columns), None)
        if post_text_col:
            mapping = dict(zip(df_posts['post_id'], df_posts[post_text_col]))
            df_batch3['context_text'] = df_batch2['post_id'].map(mapping).fillna("[TEKS UNGGAHAN KOSONG]")

    # Simpan HANYA Batch 3 ke file baru
    df_batch3.to_csv(labels_batch2_path, index=False)

    print(f"Berhasil membuat file kerja baru untuk Batch 3 di: {labels_batch2_path} (Total: {len(df_batch2)} baris)")
    print("\nPratinjau data Batch 3 (Kolom Bersih & Teks Konteks Terisi):")
    display(df_batch3[['comment_id', 'post_id', 'context_text', 'text_original', 'context_target_phrase']].head(5))
else:
    print(f"File {processed_path} tidak ditemukan.")

Berhasil membuat file kerja baru untuk Batch 3 di: data/processed/labels_student_v1(b3).csv (Total: 200 baris)

Pratinjau data Batch 3 (Kolom Bersih & Teks Konteks Terisi):


,comment_id,post_id,context_text,text_original,context_target_phrase
400,IG-P013-C019,IG-P013,,Usut-tuntas-TPPU-Dana-Syariah-Indonesia. Mafia...,
401,IG-P013-C018,IG-P013,,Help Usut-tuntas-TPPU-Dana-Syariah-Indonesia. ...,
402,IG-P013-C017,IG-P013,,"Rp2,4 T belum kembali. Aset disita baru >Rp300...",
403,IG-P013-C016,IG-P013,,Aku nggak sabar melihat penampilan seluruh pem...,
404,IG-P013-C015,IG-P013,,"Catet tanggalnya guys 25-27 September 2026, ja...",


**Memeriksa ulang 10 label acak dari 200 tersebut**

In [9]:
import pandas as pd

# Load file final
df = pd.read_csv("data/processed/labels_student_v1_(labelling_b3).csv")

# Ambil 10 sampel acak untuk diperiksa ulang (sesuai SOP)
sample_10 = df.sample(n=10, random_state=42)

print("==========================================")
print("     10 SAMPEL ACAK UNTUK DICEK ULANG     ")
print("==========================================")
for idx, row in sample_10.iterrows():
    print(f"Baris {idx} | ID: {row['comment_id']}")
    print(f"Teks : {row['text_original']}")
    print(f"Topik: {row['context_target_phrase']} | Label: {row['sentiment_label']} | Catatan: {row['annotation_note']}")
    print("-" * 50)

     10 SAMPEL ACAK UNTUK DICEK ULANG     
Baris 95 | ID: IG-P017-C005
Teks : Mudah2an Allah berkahi hidupmu ya, nduk. Jadi kelingan jaman SD dulu. Paginya berangkat sekolah sambil jualan kripik singkong di sekolah. Kalo kedodong di halaman rumah berbuah ya jualannya nambah kedondong. Sorenya keliling dusun jualan tahu isi.
Topik: pernyataan apresiasi dan doa positif | Label: Positif | Catatan: nan
--------------------------------------------------
Baris 15 | ID: IG-P013-C004
Teks : 🔥🔥🔥
Topik: reaksi emotikon | Label: Netral | Catatan: teks asli cuma emoticon
--------------------------------------------------
Baris 30 | ID: IG-P014-C035
Teks : Rajanya kemana tuh??abdi dalem juga,apa sudah gak digunakan sebagai istananya?
Topik: diskusi dan informasi mengenai kegiatan | Label: Netral | Catatan: nan
--------------------------------------------------
Baris 158 | ID: IG-P022-C021
Teks : 😍
Topik: reaksi emotikon | Label: Netral | Catatan: teks asli cuma emoticon
----------------------------

**Menghitung kemajuan dan progress**

In [12]:
import pandas as pd

# Membaca file label utama
df_labels = pd.read_csv("data/processed/labels_student_v1_(labelling_b3).csv")

# Pastikan kolom bantu bertipe string untuk menghindari error .str accessor
df_labels["sentiment_label"] = df_labels["sentiment_label"].fillna("").astype(str)
df_labels["annotation_note"] = df_labels["annotation_note"].fillna("").astype(str)

# 1. Menghitung jumlah label yang terisi (selain string kosong)
terisi_mask = (df_labels["sentiment_label"].str.strip() != "") & (df_labels["sentiment_label"].str.lower() != "nan")
total_terisi = terisi_mask.sum()
print(f"Total label terisi: {total_terisi}")

# 2. Menghitung sebaran label (Positif, Negatif, Netral)
print("\nSebaran Label:")
if total_terisi > 0:
    print(df_labels[terisi_mask]["sentiment_label"].value_counts())
else:
    print("Belum ada label yang diisi.")

# 3. Menghitung jumlah catatan RAGU
jumlah_ragu = df_labels["annotation_note"].str.contains("RAGU", case=False, na=False).sum()
print(f"\nJumlah catatan RAGU: {jumlah_ragu}")

Total label terisi: 200

Sebaran Label:
sentiment_label
Netral     155
Positif     37
Negatif      8
Name: count, dtype: int64

Jumlah catatan RAGU: 8


**Mencatat ke file cek label**

In [13]:
# Draf otomatis untuk docs/D12_cek_label.md
cek_label_d12_content = """# Hasil Pemeriksaan Ulang 10% Label Batch 3 (Hari 13)
**Tanggal Pelaksanaan:** 23 Oktober 2026

## 1. Ringkasan Pemeriksaan Ulang Batch 2 (200 Baris)
- **Total Label Batch 3 yang Diperiksa:** 200 komentar
- **Jumlah Label yang Berubah (setelah pemeriksaan sampel 10%):** 0 komentar (seluruh sampel uji konsisten dengan panduan pelabelan).
- **Persentase Perubahan:** 0.0% (0 dari 200)
- **Alasan Perubahan Utama:** Tidak ada revisi label sentimen. Fokus pemeriksaan diarahkan pada konsistensi penulisan `context_target_phrase` agar spesifik menjelaskan inti topik teks asli tanpa mencantumkan kata-kata mentah dari teks asli.

## 2. Pemeriksaan Sampel 10% (10 Label Acak)
- **Jumlah Sampel Diperiksa:** 10 komentar acak dari *batch* data.
- **Sebaran Progress & Statistik Riil:**
  - Total Terisi: 200 / 200 baris
  - Sebaran Label: Positif (37), Netral (155), Negatif (8)
  - Jumlah Catatan RAGU: 8 baris
- **Kesimpulan:** Secara umum pedoman v1 telah diterapkan dengan konsisten pada Batch 2, dengan seluruh frasa topik spesifik dan label sentimen tervalidasi dengan baik.
"""

import os
os.makedirs("docs", exist_ok=True)
with open("docs/D13_cek_label.md", "w", encoding="utf-8") as f:
    f.write(cek_label_d12_content)

print("Berhasil membuat dokumen docs/D13_cek_label.md")

Berhasil membuat dokumen docs/D13_cek_label.md


**Membuat file queries.py untuk fungsi pencarian**

In [16]:
import os

# Pastikan folder src/ sudah ada
os.makedirs("src", exist_ok=True)

queries_code = '''import pandas as pd

def search_comments(conn, keyword):
    sql = """SELECT c.comment_id, p.platform, c.published_at AS commented_at,
                c.text_original, p.text_content AS context_text
                FROM comments c JOIN posts p ON p.post_id = c.post_id
                WHERE LOWER(c.text_original) LIKE ?"""
    return pd.read_sql(sql, conn, params=[f"%{keyword.lower()}%"])
'''

with open("src/queries.py", "w", encoding="utf-8") as f:
    f.write(queries_code)

print("File src/queries.py berhasil diperbarui!")

File src/queries.py berhasil diperbarui!


**Membuat file 2_Data.py untuk halaman antarmuka pencarian di streamlit**

In [15]:
import os

# Pastikan folder app/pages/ sudah ada
os.makedirs("app/pages", exist_ok=True)

data_page_code = '''import os
import streamlit as st
import pandas as pd
import sys

# Menambahkan path src agar dapat diimpor
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '../../')))
from src.db import get_conn, init_db
from src.queries import search_comments

st.title("Pencarian Komentar & Eksplorasi Data")
st.write("Cari kata kunci pada isi komentar secara fleksibel (case-insensitive).")

init_db()

# Kotak pencarian
keyword = st.text_input("Masukkan kata kunci pencarian:", "")

if keyword:
    conn = get_conn()
    df_res = search_comments(conn, keyword)
    conn.close()

    count = len(df_res)
    st.write(f"Ditemukan {count} komentar")

    if count > 0:
        st.dataframe(df_res)
    else:
        st.info("Tidak ada komentar yang cocok dengan kata kunci tersebut.")
else:
    st.info("Silakan masukkan kata kunci di atas untuk mulai mencari.")
'''

with open("app/pages/2_Data.py", "w", encoding="utf-8") as f:
    f.write(data_page_code)

print("Berhasil membuat file app/pages/2_Data.py!")

Berhasil membuat file app/pages/2_Data.py!


**Pengecekan jumlah hasil dengan query manual**

In [21]:
import sqlite3
import pandas as pd

# 1. Hubungkan ke database SQLite (sesuaikan path jika database berada di folder data/app.db)
db_path = "data/app.db"
conn = sqlite3.connect(db_path)

# 2. Definisikan 3 kata kunci pengujian
test_keywords = ["TPPU", "FESYAR", "xyzabc999"]

# 3. Kueri SQL manual yang sama persis dengan fungsi pencarian aplikasi
sql = """
SELECT c.comment_id, p.platform, c.published_at AS commented_at,
       c.text_original, p.text_content AS context_text
FROM comments c
JOIN posts p ON p.post_id = c.post_id
WHERE LOWER(c.text_original) LIKE ?
"""

# 4. Jalankan pengujian untuk ketiga kata kunci
for kw in test_keywords:
    print(f"=" * 50)
    print(f"Hasil Kueri Manual untuk Kata Kunci: '{kw}'")
    print(f"=" * 50)

    # Eksekusi kueri dengan parameter pencarian (case-insensitive)
    df_manual = pd.read_sql(sql, conn, params=[f"%{kw.lower()}%"])

    count = len(df_manual)
    print(f"Ditemukan {count} komentar\n")

    if count > 0:
        display(df_manual.head(3)) # Tampilkan 3 data teratas sebagai sampel
    else:
        print(f"INFO: Tidak ada komentar yang cocok dengan kata kunci '{kw}' (0 komentar ditemukan).\n")

# Tutup koneksi database
conn.close()

Hasil Kueri Manual untuk Kata Kunci: 'TPPU'
Ditemukan 7 komentar



,comment_id,platform,commented_at,text_original,context_text
0,IG-P013-C018,Instagram,2026-09-21,Help Usut-tuntas-TPPU-Dana-Syariah-Indonesia. ...,
1,IG-P013-C019,Instagram,2026-09-21,Usut-tuntas-TPPU-Dana-Syariah-Indonesia. Mafia...,
2,IG-P013-C020,Instagram,2026-09-21,#KasusDSITPPUDuaKomaEmpatTriliun #LenderDSIMe...,


Hasil Kueri Manual untuk Kata Kunci: 'FESYAR'
Ditemukan 17 komentar



,comment_id,platform,commented_at,text_original,context_text
0,IG-P012-C020,Instagram,2026-09-22,Aku nggak sabar melihat penampilan seluruh pem...,
1,IG-P013-C016,Instagram,2026-09-22,Aku nggak sabar melihat penampilan seluruh pem...,
2,IG-P022-C015,Instagram,2026-09-22,"Wah, temanya menarik banget nih pembahasannya!...",


Hasil Kueri Manual untuk Kata Kunci: 'xyzabc999'
Ditemukan 0 komentar

INFO: Tidak ada komentar yang cocok dengan kata kunci 'xyzabc999' (0 komentar ditemukan).

